# 外部信息 PIT 回测准入检查

## tl;dr

在 2023-07-13 至 2026-05-26 的共同窗口内，28,049 条机会候选、全球/宏观状态和 155,132 条经筛选巨潮公告可以做**研究级增量筛选**。但当前不能把结果称为 V3 权重回测：候选集没有完整 `core_score`，历史板块归属也没有 PIT 生效区间；专业新闻历史源仍为空。生产权重继续保持 `lambda=0`。

## Context & Methods

目标是判断现有本地数据能否支持“全球状态 / 板块状态 / 个股事件”的增量辨别检验，不估计新模型，也不修改 V3。

### Key Assumptions

- 决策截止时点统一为信号日 `23:59:59 Asia/Shanghai`。
- 所有外部记录必须满足 `available_at <= decision_cutoff`；历史回放不联网。
- 巨潮公告的 `available_from` 使用平台所示发布日期的保守日末边界；同日公告不会进入同日决策。
- 共同研究窗口截至 2026-05-26，以完整巨潮回补边界为准；其后的近期数据不用于本次准入判断。
- 当前股票行业字段没有历史生效区间；即使静态映射覆盖率高，也不计为 PIT 可用。

In [1]:
import bisect
import gzip
import json
import os
import sqlite3
import statistics
import sys
from collections import Counter, defaultdict
from datetime import date, datetime, time
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from ashare_evidence.external_context_global_market_research import market_state_by_decision_date
from ashare_evidence.external_context_macro_research import macro_state_by_decision_date
from ashare_evidence.external_context_sector_market_research import sector_mapping_coverage

REPORT_DIR = REPO_ROOT / "docs/analysis/external_context_pit_readiness_2026-08-17"
RUNTIME_ROOT = Path("/Users/hernando_zhao/codex/runtime/projects/ashare-dashboard")
EXTERNAL_ROOT = RUNTIME_ROOT / "data/artifacts/external-context-personal-v1"
CANDIDATE_PATH = REPO_ROOT / "docs/research/data/ALL_UNIVERSE_OPPORTUNITY_DATASET_COVERAGE_CORRECTED_V2_2026-08-17.json.gz"
GLOBAL_PATH = REPO_ROOT / "data/research_validation_forward_20260808/external/tushare-global-market-research-20230501-20260807.json"
MACRO_PATH = REPO_ROOT / "data/research_validation_forward_20260808/external/macro-market-research-20230501-20260805.json"
SECTOR_PATH = REPO_ROOT / "data/research_validation_forward_20260808/external/tushare-sw2021-l1-sector-market-research-20230501-20260807.json"

required_paths = [CANDIDATE_PATH, GLOBAL_PATH, MACRO_PATH, SECTOR_PATH, EXTERNAL_ROOT / "cninfo-full713-plan.json"]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required inputs: {missing_paths}"

## Data

In [2]:
with gzip.open(CANDIDATE_PATH, "rt", encoding="utf-8") as handle:
    candidate_artifact = json.load(handle)
candidate_rows = candidate_artifact["rows"]
candidate_keys = [(row["signal_day"], row["symbol"]) for row in candidate_rows]
candidate_days = sorted({date.fromisoformat(row["signal_day"]) for row in candidate_rows})
candidate_symbols = {row["symbol"] for row in candidate_rows}

with (EXTERNAL_ROOT / "cninfo-full713-plan.json").open(encoding="utf-8") as handle:
    cninfo_plan = json.load(handle)
with (EXTERNAL_ROOT / "official-poc/cninfo-curation-full.json").open(encoding="utf-8") as handle:
    curation = json.load(handle)
with GLOBAL_PATH.open(encoding="utf-8") as handle:
    global_artifact = json.load(handle)
with MACRO_PATH.open(encoding="utf-8") as handle:
    macro_artifact = json.load(handle)
with SECTOR_PATH.open(encoding="utf-8") as handle:
    sector_artifact = json.load(handle)

overlap_start = max(date.fromisoformat(candidate_artifact["observed_from"]), date.fromisoformat(cninfo_plan["start_date"]))
overlap_end = min(date.fromisoformat(candidate_artifact["observed_to"]), date.fromisoformat(cninfo_plan["end_date"]))
overlap_rows = [
    row for row in candidate_rows if overlap_start <= date.fromisoformat(row["signal_day"]) <= overlap_end
]
overlap_days = sorted({date.fromisoformat(row["signal_day"]) for row in overlap_rows})
overlap_symbols = {row["symbol"] for row in overlap_rows}
plan_symbols = {row["symbol"] for row in cninfo_plan["symbols"]}

candidate_profile = {
    "candidate_rows": len(candidate_rows),
    "duplicate_candidate_keys": len(candidate_keys) - len(set(candidate_keys)),
    "candidate_bearing_days": len(candidate_days),
    "audited_days": len(candidate_artifact["daily_audits"]),
    "zero_candidate_audit_days": sorted(
        set(row["signal_day"] for row in candidate_artifact["daily_audits"])
        - set(row["signal_day"] for row in candidate_rows)
    ),
    "candidate_symbols": len(candidate_symbols),
    "overlap_from": overlap_start.isoformat(),
    "overlap_to": overlap_end.isoformat(),
    "overlap_rows": len(overlap_rows),
    "overlap_days": len(overlap_days),
    "overlap_symbols": len(overlap_symbols),
    "overlap_symbols_in_cninfo_plan": len(overlap_symbols & plan_symbols),
    "overlap_completed_labels": sum(row.get("net_return_5d") is not None for row in overlap_rows),
    "v3_soft_quality_nonzero_rows": sum(float(row.get("v3_soft_quality") or 0.0) != 0.0 for row in overlap_rows),
    "core_score_present_rows": sum(row.get("core_score") is not None for row in overlap_rows),
    "candidate_sector_field_rows": sum(
        any(row.get(field) not in (None, "") for field in ("industry_code", "industry_name", "sector_code", "sector_name"))
        for row in overlap_rows
    ),
}
candidate_profile

{'candidate_rows': 32028,
 'duplicate_candidate_keys': 0,
 'candidate_bearing_days': 749,
 'audited_days': 750,
 'zero_candidate_audit_days': ['2024-10-09'],
 'candidate_symbols': 2839,
 'overlap_from': '2023-07-13',
 'overlap_to': '2026-05-26',
 'overlap_rows': 28049,
 'overlap_days': 692,
 'overlap_symbols': 2788,
 'overlap_symbols_in_cninfo_plan': 2788,
 'overlap_completed_labels': 26144,
 'v3_soft_quality_nonzero_rows': 36,
 'core_score_present_rows': 0,
 'candidate_sector_field_rows': 0}

In [3]:
global_states = market_state_by_decision_date(global_artifact["records"], decision_dates=candidate_days)
macro_states = macro_state_by_decision_date(macro_artifact["records"], decision_dates=candidate_days)
sector_records = sector_artifact["normalized"]["records"]

def age_profile(states, days, nested_key=None):
    ages = []
    for decision_day in days:
        state = states[decision_day.isoformat()]
        items = state[nested_key] if nested_key else state
        for value in items.values():
            ages.append((decision_day - date.fromisoformat(value["observation_date"])).days)
    return {"maximum_calendar_days": max(ages), "p95_calendar_days": statistics.quantiles(ages, n=20)[18]}

market_profile = {
    "global_records": len(global_artifact["records"]),
    "global_instruments": len({row["instrument_id"] for row in global_artifact["records"]}),
    "global_duplicate_keys": len(global_artifact["records"])
    - len({(row["instrument_id"], row["trade_date"]) for row in global_artifact["records"]}),
    "global_state_days_full": len(global_states),
    "global_state_days_overlap": sum(day.isoformat() in global_states for day in overlap_days),
    "global_overlap_observation_age": age_profile(global_states, overlap_days, "instruments"),
    "global_provider_revision_id_available": bool(global_artifact["provider_revision_id_available"]),
    "macro_records": len(macro_artifact["records"]),
    "macro_series": len({row["series_id"] for row in macro_artifact["records"]}),
    "macro_duplicate_keys": len(macro_artifact["records"])
    - len({(row["series_id"], row["observation_date"]) for row in macro_artifact["records"]}),
    "macro_state_days_full": len(macro_states),
    "macro_state_days_overlap": sum(day.isoformat() in macro_states for day in overlap_days),
    "macro_overlap_observation_age": age_profile(macro_states, overlap_days),
    "macro_provider_revision_id_available": bool(macro_artifact["provider_revision_id_available"]),
    "sector_records": len(sector_records),
    "sector_series": len({row["sector_code"] for row in sector_records}),
    "sector_duplicate_keys": len(sector_records)
    - len({(row["sector_code"], row["trade_date"]) for row in sector_records}),
    "sector_provider_revision_id_available": bool(sector_artifact["provider_revision_id_available"]),
}
market_profile

{'global_records': 3246,
 'global_instruments': 4,
 'global_duplicate_keys': 0,
 'global_state_days_full': 749,
 'global_state_days_overlap': 692,
 'global_overlap_observation_age': {'maximum_calendar_days': 5,
  'p95_calendar_days': 3.0},
 'global_provider_revision_id_available': False,
 'macro_records': 5912,
 'macro_series': 7,
 'macro_duplicate_keys': 0,
 'macro_state_days_full': 749,
 'macro_state_days_overlap': 692,
 'macro_overlap_observation_age': {'maximum_calendar_days': 11,
  'p95_calendar_days': 4.0},
 'macro_provider_revision_id_available': False,
 'sector_records': 24583,
 'sector_series': 31,
 'sector_duplicate_keys': 0,
 'sector_provider_revision_id_available': False}

In [4]:
runtime_database = RUNTIME_ROOT / "data/ashare_dashboard.db"
connection = sqlite3.connect(f"file:{runtime_database}?mode=ro", uri=True)
try:
    current_profiles = {
        symbol: json.loads(profile_payload or "{}")
        for symbol, profile_payload in connection.execute("SELECT symbol, profile_payload FROM stocks")
    }
    runtime_membership_count, runtime_membership_stocks = connection.execute(
        "SELECT COUNT(*), COUNT(DISTINCT stock_id) FROM sector_memberships"
    ).fetchone()
finally:
    connection.close()

current_industries = [current_profiles.get(symbol, {}).get("industry") for symbol in candidate_symbols]
static_mapping = sector_mapping_coverage([value for value in current_industries if value])
sector_join_profile = {
    "runtime_membership_rows": runtime_membership_count,
    "runtime_membership_stocks": runtime_membership_stocks,
    "candidate_symbols_with_current_industry": sum(bool(value) for value in current_industries),
    "current_static_sw_l1_mapping_rate": static_mapping["mapped_row_rate"],
    "pit_effective_membership_available": False,
    "why_blocked": "current stock profile has no historical effective_from/effective_to lineage",
}
sector_join_profile

{'runtime_membership_rows': 7,
 'runtime_membership_stocks': 4,
 'candidate_symbols_with_current_industry': 2839,
 'current_static_sw_l1_mapping_rate': 0.9992955265938711,
 'pit_effective_membership_available': False,
 'why_blocked': 'current stock profile has no historical effective_from/effective_to lineage'}

## Results

In [5]:
excluded_versions = {
    (str(row["normalized_event_id"]), str(row["revision_id"]))
    for row in curation["excluded_event_versions"]
}
event_times_by_code = defaultdict(list)
event_categories = Counter()
event_pairs = set()
event_duplicates = 0
event_missing = Counter()
source_event_count = 0
curated_event_count = 0

for entry in os.scandir(EXTERNAL_ROOT / "pit/records"):
    if not entry.name.endswith(".json"):
        continue
    with open(entry.path, encoding="utf-8") as handle:
        row = json.load(handle)
    event_id = str(row.get("normalized_event_id") or "")
    if not event_id.startswith("cninfo"):
        continue
    source_event_count += 1
    pair = (event_id, str(row.get("knowledge_version") or ""))
    event_duplicates += pair in event_pairs
    event_pairs.add(pair)
    if pair in excluded_versions:
        continue
    curated_event_count += 1
    feature_value = row.get("feature_value") or {}
    available_from = row.get("available_from")
    security_code = str(feature_value.get("sec_code") or "")
    category = str(feature_value.get("materiality_category") or "")
    required = {
        "event_id": event_id,
        "revision_id": pair[1],
        "available_from": available_from,
        "security_code": security_code,
        "category": category,
        "availability_evidence_ref": row.get("availability_evidence_ref"),
    }
    for field, value in required.items():
        if not value:
            event_missing[field] += 1
    if available_from:
        available_at = datetime.fromisoformat(str(available_from))
        if available_at.tzinfo is None:
            event_missing["available_from_timezone"] += 1
        event_times_by_code[security_code].append(available_at)
    event_categories[category] += 1

for values in event_times_by_code.values():
    values.sort()

shanghai = ZoneInfo("Asia/Shanghai")
event_windows = (1, 5, 20)
event_hits = Counter()
event_hit_symbols = set()
segment_rows = defaultdict(list)
segment_definitions = {
    "tuning": (date.min, date(2025, 5, 26)),
    "validation": (date(2025, 5, 27), date(2025, 11, 26)),
    "final": (date(2025, 11, 27), overlap_end),
}

for row in overlap_rows:
    signal_day = date.fromisoformat(row["signal_day"])
    cutoff = datetime.combine(signal_day, time(23, 59, 59), tzinfo=shanghai)
    timestamps = event_times_by_code.get(row["symbol"].split(".")[0], [])
    right = bisect.bisect_right(timestamps, cutoff)
    for window_days in event_windows:
        window_start = datetime.combine(date.fromordinal(signal_day.toordinal() - window_days), time(23, 59, 59), tzinfo=shanghai)
        left = bisect.bisect_right(timestamps, window_start)
        if right > left:
            event_hits[window_days] += 1
            if window_days == 20:
                event_hit_symbols.add(row["symbol"])
    for segment, (start, end) in segment_definitions.items():
        if start <= signal_day <= end:
            segment_rows[segment].append(row)
            break

event_profile = {
    "source_events": source_event_count,
    "curated_events": curated_event_count,
    "curated_symbols": len(event_times_by_code),
    "duplicate_event_versions": event_duplicates,
    "missing_required_fields": dict(event_missing),
    "category_counts": dict(sorted(event_categories.items())),
    "candidate_row_event_hit_rates": {
        f"prior_{window_days}_calendar_days": event_hits[window_days] / len(overlap_rows)
        for window_days in event_windows
    },
    "candidate_symbols_with_prior_20d_event": len(event_hit_symbols),
}

segment_profile = []
for segment, rows in segment_rows.items():
    completed = [row for row in rows if row.get("net_return_5d") is not None]
    segment_profile.append(
        {
            "segment": segment,
            "candidate_rows": len(rows),
            "candidate_days": len({row["signal_day"] for row in rows}),
            "candidate_symbols": len({row["symbol"] for row in rows}),
            "completed_labels": len(completed),
            "label_completion_rate": len(completed) / len(rows),
        }
    )

event_profile, segment_profile

({'source_events': 155518,
  'curated_events': 155132,
  'curated_symbols': 3007,
  'duplicate_event_versions': 0,
  'missing_required_fields': {},
  'category_counts': {'capital_and_ownership': 57115,
   'financial_performance_and_distribution': 48949,
   'financing_and_mna': 13204,
   'management_change': 3065,
   'material_operations': 11615,
   'risk_enforcement_and_correction': 21184},
  'candidate_row_event_hit_rates': {'prior_1_calendar_days': 0.04367357125031195,
   'prior_5_calendar_days': 0.21002531284537773,
   'prior_20_calendar_days': 0.6486149238832044},
  'candidate_symbols_with_prior_20d_event': 2648},
 [{'segment': 'tuning',
   'candidate_rows': 18433,
   'candidate_days': 450,
   'candidate_symbols': 2611,
   'completed_labels': 17200,
   'label_completion_rate': 0.9331090978137037},
  {'segment': 'validation',
   'candidate_rows': 3955,
   'candidate_days': 125,
   'candidate_symbols': 1518,
   'completed_labels': 3678,
   'label_completion_rate': 0.9299620733249052}

In [6]:
logical_storage_bytes = sum(path.stat().st_size for path in EXTERNAL_ROOT.rglob("*") if path.is_file())
allocated_blocks = 0
for path in EXTERNAL_ROOT.rglob("*"):
    if path.is_file():
        allocated_blocks += path.stat().st_blocks * 512

channel_readiness = [
    {
        "component": "机会候选与股票路径",
        "status": "partial",
        "research_use": "可做机会头机制筛选",
        "blocker": "无完整 V3 core_score；历史 ST 非 PIT",
    },
    {
        "component": "全球 / 宏观状态",
        "status": "provisional_ready",
        "research_use": "可做共同窗口风险预算消融",
        "blocker": "单一低成本供应链且无 revision/vintage，不可晋级生产",
    },
    {
        "component": "巨潮个股官方事件",
        "status": "provisional_ready",
        "research_use": "可做共同窗口个股事件残差筛选",
        "blocker": "历史修订链不完整；仅标题与元数据",
    },
    {
        "component": "申万板块状态与个股归属",
        "status": "blocked",
        "research_use": "板块指数本身可回放",
        "blocker": "缺股票→板块 PIT membership，当前静态映射禁止历史回填",
    },
    {
        "component": "专业新闻",
        "status": "blocked",
        "research_use": "无历史样本",
        "blocker": "没有合格历史 feed / 摘要 / 修订链",
    },
    {
        "component": "完整 V3 外部权重回测",
        "status": "blocked",
        "research_use": "不得宣称已启动",
        "blocker": "core_score、PIT 板块归属、独立新闻源与生产级 revision 均未齐",
    },
]

summary = {
    "artifact_type": "external_context_pit_readiness_audit",
    "schema_version": "external_context_pit_readiness_audit.v1",
    "as_of": "2026-08-17",
    "decision_cutoff_convention": "signal_day_23_59_59_asia_shanghai",
    "network_used": False,
    "v3_signal_changed": False,
    "backtest_executed": False,
    "overall_assessment": "partial_provisional_research_ready_not_v3_weight_backtest_ready",
    "candidate_profile": candidate_profile,
    "market_profile": market_profile,
    "sector_join_profile": sector_join_profile,
    "event_profile": event_profile,
    "segment_profile": segment_profile,
    "storage_profile": {
        "logical_bytes": logical_storage_bytes,
        "allocated_bytes": allocated_blocks,
        "file_count": sum(1 for path in EXTERNAL_ROOT.rglob("*") if path.is_file()),
        "user_exception": "uncompressed_small_file_layout_accepted_in_prior_turn",
    },
    "channel_readiness": channel_readiness,
    "allowed_next_experiment": {
        "scope": "opportunity_head_mechanism_screen_only",
        "window": [overlap_start.isoformat(), overlap_end.isoformat()],
        "arms": ["stock_path_control", "global_macro_only", "official_event_only", "global_macro_plus_official_event"],
        "forbidden_claims": ["V3_improved", "production_ready", "sector_increment_validated", "professional_news_validated"],
        "must_precede_exact_v3_test": [
            "materialize_full_candidate_core_score",
            "acquire_or_construct_effective_dated_PIT_sector_membership",
            "freeze_feature_freshness_and_missingness_contract",
        ],
    },
}

summary_path = REPORT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

pd.DataFrame(channel_readiness)

,component,status,research_use,blocker
0,机会候选与股票路径,partial,可做机会头机制筛选,无完整 V3 core_score；历史 ST 非 PIT
1,全球 / 宏观状态,provisional_ready,可做共同窗口风险预算消融,单一低成本供应链且无 revision/vintage，不可晋级生产
2,巨潮个股官方事件,provisional_ready,可做共同窗口个股事件残差筛选,历史修订链不完整；仅标题与元数据
3,申万板块状态与个股归属,blocked,板块指数本身可回放,缺股票→板块 PIT membership，当前静态映射禁止历史回填
4,专业新闻,blocked,无历史样本,没有合格历史 feed / 摘要 / 修订链
5,完整 V3 外部权重回测,blocked,不得宣称已启动,core_score、PIT 板块归属、独立新闻源与生产级 revision 均未齐


In [7]:
pd.DataFrame(segment_profile)

,segment,candidate_rows,candidate_days,candidate_symbols,completed_labels,label_completion_rate
0,tuning,18433,450,2611,17200,0.933109
1,validation,3955,125,1518,3678,0.929962
2,final,5661,117,1834,5266,0.930224


## Takeaways

1. 数据基础不再是“全阻塞”：全球/宏观和巨潮公告可在共同窗口做研究级、离线、PIT 机制筛选。
2. 当前 32,028 条数据是热点机会头候选，不是完整 V3 候选矩阵。共同窗口内仅 36 条 `v3_soft_quality` 非零，且没有 `core_score`；因此任何结果只能说明外部信息是否改善机会头辨别，不能说明 V3 已优化。
3. 31 条申万一级行业指数序列本身完整，但股票行业只存在当前静态标签。99.93% 的静态映射率不能替代历史 PIT 生效区间。
4. 巨潮公告的来源覆盖与事件密度足够做筛选：共同窗口候选全部在采集计划内，约 64.9% 的候选在此前 20 个自然日内有至少一条保留事件。无事件必须与“渠道缺失”分开编码。
5. 下一步应先跑四臂机会头消融；若出现稳定新增辨别力，再补 `core_score` 和 PIT 板块归属，进入精确 V3 残差权重检验。专业新闻仍保持延后。